## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
%pip install -U pypdf gradio

Note: you may need to restart the kernel to use updated packages.


In [2]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

/home/bruno/Workspace/udemy-ai-agentic-track/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

   
Contact
+5562985171987 (Mobile)
brunoconterato@gmail.com
www.linkedin.com/in/
brunoconterato (LinkedIn)
github.com/brunoconterato
(Personal)
Top Skills
LangChain
Natural Language Processing
Reinforcement Learning
Languages
Português (Native or Bilingual)
Inglês (Full Professional)
Honors-Awards
Winner 1º Hackthon Grupo Rio
Quente
Uber Urban Mobility Challenge
Bruno Conterato
Machine Learning Developer | AI Engineer
Goiânia, Goiás, Brazil
Summary
Bruno Conterato is a seasoned Machine Learning Engineer with
over 6 years of experience in software development and consulting.
He excels in designing and implementing intelligent algorithms,
with notable projects including vehicle damage detection and
debt collection systems. His expertise spans a wide range of
technologies, including machine learning frameworks, NLP models,
and cloud platforms.
With a strong foundation in MLOps practices, Bruno is adept at
creating, deploying, and monitoring machine learning models.
He has hands-on experi

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Bruno Conterato"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Bruno Conterato. You are answering questions on Bruno Conterato's website, particularly questions related to Bruno Conterato's career, background, skills and experience. Your responsibility is to represent Bruno Conterato for interactions on the website as faithfully as possible. You are given a summary of Bruno Conterato's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nDeep Learning and Applied AI Engineer with experience in model development, adaptation of existing models, fine-tuning, and building\ndata-driven intelligent applications.\nPrimarily focused on deep learning, NLP, LLMs, RAG, and conversational agents, bridging modeling, backend engineering, and product\ndevelopment to transform research and prototypes into production-ready solutions.\nStrong interest in foundatio

In [10]:
def chat(message, history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [11]:

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [12]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [13]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [14]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# import os
# gemini = OpenAI(
#     api_key=os.getenv("GOOGLE_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

In [19]:
from openai.types.chat import ChatCompletionMessageParam

# def evaluate(reply, message, history) -> Evaluation:

#     messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
#     response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
#     return response.choices[0].message.parsed


def evaluate(reply, message, history) -> Evaluation:

    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    response = openai.beta.chat.completions.parse(
        model="gpt-5-nano",
        messages=messages,
        response_format=Evaluation,
    )

    parsed = response.choices[0].message.parsed
    if parsed is None:
        raise ValueError("Evaluator returned no structured output.")

    return parsed

In [20]:
messages = [{"role": "system", "content": system_prompt}] + [
    {"role": "user", "content": "do you hold a patent?"}
]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [21]:
reply

'As of my last update, I do not hold any patents. My focus has primarily been on developing and implementing machine learning models, engaging in research, and creating intelligent applications. If you have any specific inquiries about my projects or areas of expertise, feel free to ask!'

In [22]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The agent provides a direct, honest answer (no patents) and stays professional, aligning with the Bruno Conterato persona. It’s concise and invites further discussion about projects or expertise. Minor improvement: consider removing time-based phrasing like "As of my last update" and adding a sentence highlighting willingness to discuss notable projects or IP-related topics if relevant.')

In [23]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [24]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
A resposta é ilegível e não clara. Ela usa um texto que parece Pig Latin/pseudolingual e não responde diretamente à pergunta sobre patentes. Em termos profissionais, especialmente para um site de apresentação, a mensagem deveria ser clara, em português (ou outro idioma acordado), e indicar diretamente se há patentes registradas ou não, além de oferecer detalhes relevantes (portfólio, exemplos de IP potencial, demonstrações). Sugiro responder de forma direta, por exemplo: “Atualmente não possuo patentes registradas. Posso compartilhar detalhes de projetos e demonstrações técnicas relevantes em IA/NLP, bem como links para meu portfólio e GitHub. Se houver interesse, posso discutir oportunidades de proteção de IP em projetos futuros.”
